# 04 — Ablation đợt 2 (TIER_B) theo giao thức E1 — bổ sung cho R3.3

Notebook này giống hệt `03_baselines_ablation_E1.ipynb`, chỉ khác cấu hình: chạy 7 biến thể còn lại
**JA, JM_no_adaptive, A, J, M, MA, GRU** (5 seed, `COORD_MODE = 'pixel'`). Kết quả ghi vào cùng thư mục
`results_v2/pixel/ablation/` với đợt 1, nên bảng tổng hợp cuối notebook gồm đủ 15 biến thể (đợt 1 không chạy lại).
Biến thể `full` dùng lại kết quả E1 của notebook 02.

Ghi chú: ở biến thể một luồng (J, M, A), nhánh skip vẫn bật nhưng chỉ lấy đặc trưng sớm của chính luồng đó.

---

## Thông tin chung (như notebook 03)

Dùng cùng dữ liệu (`custom_all_v2.npz`), cùng cách chia E1 (3 fold theo người, val chọn theo nhóm video),
cùng siêu tham số, augmentation, loss, sampler và ngưỡng Youden như `02_train_protocols_v2.ipynb`.
Chỉ thay mô hình / luồng đầu vào. Mỗi biến thể chạy 3 fold × 5 seed.

| Biến thể | Mô tả | Trả lời |
|---|---|---|
| `full` | A3S-STGCN đầy đủ (J+M+A, adaptive, skip) — **dùng lại kết quả E1 của notebook 02** | – |
| `stgcn` | ST-GCN: chỉ luồng Joint, đồ thị tĩnh, không skip (cùng backbone) | R1.4, R2.5, R3.4 |
| `2s_agcn` | 2s-AGCN [28]: luồng Joint + Bone, A + B (học được) + C (attention theo mẫu), cộng điểm 2 luồng | R1.2, R2.5, R3.4 |
| `lstm`, `gru` | Mạng hồi quy 2 lớp trên tọa độ khớp | R2.5 |
| `JM` | Bỏ luồng Angle (J+M, có adaptive, có skip) | R3.3 |
| `no_skip` | Bỏ cross-stream skip | R3.2, R3.3 |
| `no_adaptive` | Bỏ ma trận kề học được | R3.3 |
| `JMB` | Thay Angle bằng vector xương thô (dx, dy) chưa chuẩn hóa | R1.3, R2.7 |
| `J`, `M`, `A`, `JA`, `MA` | Các tổ hợp luồng còn lại | R3.3 |
| `JM_no_adaptive` | Bỏ cả Angle lẫn adaptive (giống dòng cũ trong Bảng 7) | R3.3 |

Lưu ý: luồng Angle `(cos θ, sin θ)` chính là vector xương đã chuẩn hóa độ dài, nên biến thể "normalized bone"
trùng với `full`; `JMB` là phép so sánh có ý nghĩa (xương chưa chuẩn hóa).

Kết quả lưu ở `results_v2/<COORD_MODE>/ablation/<biến thể>/E1/<fold>/seed_<n>/`; chạy lại sẽ bỏ qua lượt đã xong.
Cuối notebook có bảng tổng hợp, số tham số, thời gian suy luận CPU, kiểm định McNemar và Wilcoxon so với `full`.

**Runtime**: GPU. Đợt này 7 biến thể × 3 fold × 5 seed = 105 lượt (khoảng một nửa thời gian đợt 1). Colab bị ngắt thì chạy lại, các lượt đã xong sẽ được bỏ qua.

## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 2. Cấu hình

In [ ]:
"""
train_mydata_multiseed.py
=========================
Three-Stream Adaptive STGCN — Multi-Seed Stability Evaluation

FIXES APPLIED:
  [FIX-A] mixup_data: ưu tiên mix cùng class (same-label mixup) để tránh
          làm mờ decision boundary khi dữ liệu mất cân bằng.
  [FIX-B] FocalLoss alpha_w: đồng bộ với sampler — dùng weight theo 4
          categories thay vì gộp binary, giúp loss và sampler nhất quán.
  [FIX-C] build_weighted_sampler: thêm log tỷ lệ flip/base trong train set
          để theo dõi skew do flip 100% vào train.
"""
import copy
import json
import math
import os
import random
import warnings

import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from matplotlib.colors import LinearSegmentedColormap
from sklearn.metrics import (accuracy_score, auc, confusion_matrix,
                             precision_recall_curve, precision_recall_fscore_support,
                             roc_auc_score, roc_curve)
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler

warnings.filterwarnings('ignore')

SEEDS      = [42, 123, 2026, 777, 31415]
SPLIT_SEED = SEEDS[0]   # cố định cách chọn val (và split E0) giữa các seed

ALL_NPZ         = '/content/drive/MyDrive/FallDetection_NEW/custom_all_v2.npz'
BASE_RESULTS    = '/content/drive/MyDrive/FallDetection_NEW/results_v2'

FALL_CATEGORIES = ["Lying_Falls", "Sitting_Falls", "Standing_Falls"]
ALL_CATEGORIES  = ["non_fall"] + FALL_CATEGORIES

TRAIN_RATIO = 0.70
VAL_RATIO   = 0.15
TEST_RATIO  = 0.15

NUM_CLASSES   = 2
IN_CHANNELS   = 3
NUM_KEYPOINTS = 14
SEQ_LEN       = 60

CATEGORY_LABEL = {
    'non_fall':       0,
    'Lying_Falls':    1,
    'Sitting_Falls':  2,
    'Standing_Falls': 3,
}

STREAM_CHANNELS = [24, 24, 24, 48, 48, 48, 96, 96, 96]
STREAM_STRIDES  = [ 1,  1,  1,  2,  1,  1,   2,   1,   1]
TCN_KERNEL      = 9
SKIP_BLOCK_IDX  = 2
DROPOUT         = 0.55
DROP_PATH_RATE  = 0.10
CONF_THRESH     = 0.5
FOCAL_GAMMA     = 0.85
LABEL_SMOOTHING = 0.08
FLIP_PAIRS      = [(1,2),(3,4),(5,6),(7,8),(9,10),(11,12)]

BATCH_SIZE       = 32
EPOCHS           = 150
LEARNING_RATE    = 1.2e-4
WEIGHT_DECAY     = 5.0e-3
WARMUP_EPOCHS    = 10
MIN_EPOCHS       = 30
PLATEAU_PATIENCE = 15
PLATEAU_DELTA    = 1e-4
MIXUP_ALPHA      = 0.35
INFERENCE_THR    = 0.65
FALL_ALPHA_BOOST = 0.85
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

import pandas as pd
from collections import defaultdict

# ── Cấu hình mới cho bản sửa ────────────────────────────────────────────────
# (notebook này chỉ dùng giao thức E1)
COORD_MODE       = 'pixel'              # ĐẶT GIỐNG cấu hình chính đã chọn sau khi so pixel/unit              # 'pixel' = như bài báo; 'unit' = chia x/img_w, y/img_h (ablation R2.8)
VIDEO_AGG        = 'max'                # gộp xác suất các cửa sổ thành xác suất của video
FALL_ALPHA_BOOST = FALL_ALPHA_BOOST     # giữ nguyên
RESULTS_DIR      = os.path.join(BASE_RESULTS, COORD_MODE)   # kết quả mỗi chế độ tọa độ để riêng
os.makedirs(RESULTS_DIR, exist_ok=True)
print('Device:', DEVICE, '| torch', torch.__version__)

# ── Biến thể cần chạy ───────────────────────────────────────────────────────
TIER_A = ['stgcn', '2s_agcn', 'lstm', 'JM', 'no_skip', 'no_adaptive', 'JMB']      # bắt buộc
TIER_B = ['JA', 'JM_no_adaptive', 'A', 'J', 'M', 'MA', 'gru']                     # đợt 2 — xếp theo mức ưu tiên
VARIANTS_TO_RUN = TIER_B          # notebook 04: chạy đợt 2
SEEDS_TIER_B    = SEEDS           # có thể giảm, ví dụ [42, 123, 2026], nếu thiếu thời gian
ABL_DIR = os.path.join(RESULTS_DIR, 'ablation')
os.makedirs(ABL_DIR, exist_ok=True)

## 3. Nạp dữ liệu v2

In [ ]:
def load_training_data_v2(npz_path=ALL_NPZ):
    raw = np.load(npz_path, allow_pickle=False)
    need = ['X_joint', 'y', 'img_w', 'img_h', 'subject', 'view', 'category', 'video_id',
            'window_idx', 'group_id', 'nonfall_type', 'orientation', 'is_flip']
    miss = [k for k in need if k not in raw.files]
    if miss:
        raise KeyError(f'Thiếu khóa {miss} — hãy chạy 01_extract_keypoints_v2.ipynb')
    d = {k: raw[k] for k in need}
    d['X_joint'] = d['X_joint'].astype(np.float32)
    d['y'] = d['y'].astype(np.int64)
    for k in ['img_w', 'img_h', 'video_id', 'window_idx', 'is_flip']:
        d[k] = d[k].astype(np.int64)
    assert d['X_joint'].shape[1:] == (SEQ_LEN, NUM_KEYPOINTS, IN_CHANNELS)
    assert (d['is_flip'] == 0).all(), 'Dữ liệu v2 không được chứa bản lật dựng sẵn'
    return d

data = load_training_data_v2()
meta = pd.DataFrame({k: data[k] for k in ['y', 'subject', 'view', 'category', 'video_id',
                                           'nonfall_type', 'orientation']})
print(f"Cửa sổ: {len(meta)} | video: {meta.video_id.nunique()} | người: {sorted(meta.subject.unique())}")
display(pd.crosstab([meta.subject, meta.category], meta.view, margins=True))

## 4. Tiện ích, scheduler, Focal Loss, split

In [ ]:
def set_global_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark     = False
    os.environ['PYTHONHASHSEED'] = str(seed)


def log(msg, level="INFO"):
    tag = {"INFO": "ℹ️ ", "GOOD": "✅", "WARN": "⚠️ ", "FAIL": "🔴"}.get(level, "")
    print(f"{tag} {msg}")

def sep(char="─", n=68):
    print(char * n)

In [ ]:
def split_indices_legacy(y, subject_arr, is_flip_arr, category_arr,
                  train_ratio=TRAIN_RATIO, val_ratio=VAL_RATIO,
                  seed=42):
    rng = np.random.default_rng(seed)
    all_subjects = sorted(np.unique(subject_arr))
    all_cats     = sorted(np.unique(category_arr))
    train_idx, val_idx, test_idx = [], [], []

    for subj in all_subjects:
        for cat in all_cats:
            mask_base = (
                (subject_arr == subj) &
                (category_arr == cat) &
                (is_flip_arr == 0)
            )
            base_ids = np.where(mask_base)[0]
            if len(base_ids) == 0:
                continue

            perm     = rng.permutation(len(base_ids))
            base_ids = base_ids[perm]

            n_total      = len(base_ids)
            n_val        = max(1, round(n_total * val_ratio))
            n_test       = max(1, round(n_total * TEST_RATIO))
            n_test       = min(n_test,  n_total - n_val - 1)
            n_val        = min(n_val,   n_total - n_test - 1)
            n_train_base = n_total - n_val - n_test

            tr_base = base_ids[:n_train_base]
            va_ids  = base_ids[n_train_base: n_train_base + n_val]
            te_ids  = base_ids[n_train_base + n_val:]

            val_idx.extend(va_ids.tolist())
            test_idx.extend(te_ids.tolist())

            mask_flip = (
                (subject_arr == subj) &
                (category_arr == cat) &
                (is_flip_arr == 1)
            )
            flip_ids = np.where(mask_flip)[0]
            train_idx.extend(tr_base.tolist())
            train_idx.extend(flip_ids.tolist())

    return (np.array(train_idx, dtype=np.int64),
            np.array(val_idx,   dtype=np.int64),
            np.array(test_idx,  dtype=np.int64))


def build_scheduler(optimizer, warmup_epochs, total_epochs):
    lr_min = 1e-6
    def lr_lambda(epoch):
        if epoch < warmup_epochs:
            return (epoch + 1) / warmup_epochs
        progress  = (epoch - warmup_epochs) / max(1, total_epochs - warmup_epochs)
        cosine    = 0.5 * (1.0 + math.cos(math.pi * progress))
        min_ratio = lr_min / LEARNING_RATE
        return min_ratio + (1.0 - min_ratio) * cosine
    return torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)


class FocalLoss(nn.Module):
    """
    [FIX-B] alpha nhận vào tensor 4 phần tử (theo CATEGORY_LABEL),
    sau đó map về binary khi tính loss để đồng bộ với sampler.
    Nếu alpha_mode='binary' thì giữ nguyên hành vi cũ (2 phần tử).
    """
    def __init__(self, gamma=FOCAL_GAMMA, alpha=None, reduction='mean',
                 label_smoothing=LABEL_SMOOTHING, alpha_mode='binary'):
        super().__init__()
        self.gamma      = gamma
        self.alpha      = alpha          # tensor shape (2,) — binary
        self.reduction  = reduction
        self.ls         = label_smoothing
        self.alpha_mode = alpha_mode

    def forward(self, logits, targets):
        n_cls = logits.size(1)
        log_p = F.log_softmax(logits, dim=1)
        p     = log_p.exp()
        pt    = p.gather(1, targets.unsqueeze(1)).squeeze(1)
        fw    = (1.0 - pt) ** self.gamma
        if self.alpha is not None:
            fw = self.alpha[targets] * fw
        if self.ls > 0:
            smooth = torch.full_like(logits, self.ls / n_cls)
            smooth.scatter_(1, targets.unsqueeze(1), 1.0 - self.ls + self.ls / n_cls)
            ce = -(smooth * log_p).sum(dim=1)
        else:
            ce = -log_p.gather(1, targets.unsqueeze(1)).squeeze(1)
        loss = fw * ce
        return loss.mean() if self.reduction == 'mean' else loss.sum()

In [ ]:
def group_val_split(pool_idx, data, seed, val_ratio=VAL_RATIO):
    # Chia pool thành train/val theo group_id, phân tầng theo category
    rng = np.random.default_rng(seed)
    cat, grp = data['category'], data['group_id']
    val_groups = set()
    for c in sorted(np.unique(cat[pool_idx])):
        groups = np.array(sorted(set(grp[pool_idx][cat[pool_idx] == c])))
        if len(groups) < 2:
            continue
        n_val = max(1, int(round(len(groups) * val_ratio)))
        val_groups.update(rng.permutation(groups)[:n_val].tolist())
    is_val = np.isin(grp[pool_idx], list(val_groups))
    return pool_idx[~is_val], pool_idx[is_val]


def make_folds(protocol, data):
    # Trả về list (fold_name, train_idx, val_idx, test_idx) trên chỉ số của `data`
    subj, view = data['subject'], data['view']
    all_idx = np.arange(len(data['y']))
    folds = []
    if protocol == 'E1':
        for s in sorted(np.unique(subj)):
            test_idx = all_idx[subj == s]
            tr, va = group_val_split(all_idx[subj != s], data, SPLIT_SEED)
            folds.append((f'test_{s}', tr, va, test_idx))
    elif protocol == 'E3':
        for s in sorted(np.unique(subj)):
            for a, b in [('cam1', 'cam2'), ('cam2', 'cam1')]:
                pool = all_idx[(subj != s) & (view == a)]
                test_idx = all_idx[(subj == s) & (view == b)]
                tr, va = group_val_split(pool, data, SPLIT_SEED)
                folds.append((f'{a}_to_{b}_test_{s}', tr, va, test_idx))
    else:
        raise ValueError(protocol)
    return folds


def assert_no_leak(protocol, data, tr, va, te):
    vid = data['video_id']
    a, b, c = set(vid[tr]), set(vid[va]), set(vid[te])
    assert not (a & b) and not (a & c) and not (b & c), 'Một video nằm ở hai tập!'
    if protocol in ('E1', 'E3'):
        s_tr = set(data['subject'][np.r_[tr, va]]); s_te = set(data['subject'][te])
        assert not (s_tr & s_te), f'Người {s_tr & s_te} ở cả train và test!'
        g_tr = set(data['group_id'][np.r_[tr, va]]); g_te = set(data['group_id'][te])
        assert not (g_tr & g_te), 'Một nhóm (cặp cam) ở cả train và test!'
    if protocol == 'E3':
        v_tr = set(data['view'][np.r_[tr, va]]); v_te = set(data['view'][te])
        assert not (v_tr & v_te), 'Góc quay train và test trùng nhau!'


def build_legacy_E0(data):
    # Dựng lại đúng giao thức cũ: thêm bản lật của mọi cửa sổ, subject='all', chia bằng split_indices_legacy
    n = len(data['y'])
    flipped = np.stack([aug_hflip(data['X_joint'][i], int(data['img_w'][i])) for i in range(n)])
    d = {k: np.concatenate([v, v]) for k, v in data.items() if k != 'X_joint'}
    d['X_joint'] = np.concatenate([data['X_joint'], flipped]).astype(np.float32)
    d['is_flip'] = np.r_[np.zeros(n, np.int64), np.ones(n, np.int64)]
    subj_all = np.array(['all'] * (2 * n))
    tr, va, te = split_indices_legacy(d['y'], subj_all, d['is_flip'], d['category'], seed=SPLIT_SEED)
    return d, [('random_window_split', tr, va, te)]


def describe_fold(data, tr, va, te):
    f = lambda idx: (f"{len(idx)} win / {len(set(data['video_id'][idx]))} vid / "
                     f"fall={int(data['y'][idx].sum())} / subj={sorted(map(str, set(data['subject'][idx])))} / "
                     f"view={sorted(map(str, set(data['view'][idx])))}")
    return f"train: {f(tr)}\n    val:   {f(va)}\n    test:  {f(te)}"

## 5. Mô hình

Mã A3S-STGCN giữ nguyên, chỉ thêm 2 công tắc: `adaptive` (có/không `A_adapt`) và chọn luồng / skip qua
`MultiStreamSTGCN`. Với `streams=('J','M','T'), adaptive=True, skip=True` mô hình trùng hoàn toàn với bản gốc
(cùng số tham số). Sau đó là cài đặt 2s-AGCN và baseline hồi quy.

In [ ]:
class DropPath(nn.Module):
    def __init__(self, drop_prob=0.0):
        super().__init__(); self.drop_prob = drop_prob

    def forward(self, x):
        if not self.training or self.drop_prob == 0.0:
            return x
        keep  = 1.0 - self.drop_prob
        shape = (x.shape[0],) + (1,) * (x.ndim - 1)
        mask  = torch.empty(shape, dtype=x.dtype, device=x.device).bernoulli_(keep) / keep
        return x * mask


class SkeletonGraph:
    NUM_PARTITIONS = 3
    def __init__(self, num_nodes=NUM_KEYPOINTS):
        self.num_nodes = num_nodes
        self.neighbor_link = [
            (0,1),(0,2),(1,2),(1,3),(3,5),(2,4),(4,6),
            (1,7),(2,8),(7,8),(7,9),(9,11),(8,10),(10,12),
            (7,13),(8,13),
        ]
        self.HIP_DIST = {13:0,7:1,8:1,1:2,2:2,9:2,10:2,
                         0:3,3:3,4:3,11:3,12:3,5:4,6:4}
        self.A = self._build_adjacency()

    def _build_adjacency(self):
        K, V = self.NUM_PARTITIONS, self.num_nodes
        A    = np.zeros((K, V, V), dtype=np.float32)
        for i in range(V): A[0,i,i] = 1.0
        for i, j in self.neighbor_link:
            di, dj = self.HIP_DIST[i], self.HIP_DIST[j]
            if   di < dj: A[1,j,i]=1; A[2,i,j]=1
            elif dj < di: A[1,i,j]=1; A[2,j,i]=1
            else:         A[1,i,j]=A[1,j,i]=1
        for k in range(K): A[k] = self._sym_norm(A[k])
        return A

    @staticmethod
    def _sym_norm(A):
        A = A + np.eye(A.shape[0])
        d = A.sum(axis=1)
        d_inv = np.where(d > 0, d**-0.5, 0.0)
        D = np.diag(d_inv)
        return D @ A @ D


class AdaptiveSpatialGraphConv(nn.Module):
    def __init__(self, in_ch, out_ch, num_nodes, num_partitions, adaptive=True):
        super().__init__()
        self.K       = num_partitions
        self.conv    = nn.Conv2d(in_ch, out_ch * num_partitions, kernel_size=1)
        self.A_adapt = (nn.Parameter(torch.zeros(num_partitions, num_nodes, num_nodes))
                        if adaptive else None)

    def forward(self, x, A_static):
        A_eff = A_static + torch.tanh(self.A_adapt) if self.A_adapt is not None else A_static
        N,_,T,V = x.shape
        x = self.conv(x).view(N, self.K, -1, T, V)
        return torch.einsum('kvw,nkctv->nctw', A_eff, x)


class TemporalConv(nn.Module):
    def __init__(self, channels, kernel_size=9, stride=1):
        super().__init__()
        pad = (kernel_size - 1) // 2
        self.conv = nn.Conv2d(channels, channels, (kernel_size,1),(stride,1),(pad,0))
        self.bn = nn.BatchNorm2d(channels)
        self.relu = nn.ReLU(inplace=True)
        self.tmp_maxpool = nn.MaxPool2d((kernel_size,1),(stride,1),(pad,0))

    def forward(self, x):
        res = self.tmp_maxpool(x)
        x   = self.relu(self.bn(self.conv(x)))
        return x + res


class STGCNBlock(nn.Module):
    def __init__(self, in_ch, out_ch, num_nodes, num_partitions, stride=1, drop_path_rate=0.0, adaptive=True):
        super().__init__()
        self.gcn      = AdaptiveSpatialGraphConv(in_ch, out_ch, num_nodes, num_partitions, adaptive)
        self.gcn_bn   = nn.BatchNorm2d(out_ch)
        self.res_conv = (nn.Sequential(nn.Conv2d(in_ch, out_ch, 1), nn.BatchNorm2d(out_ch))
                         if in_ch != out_ch else nn.Identity())
        self.sp_pool  = nn.MaxPool2d((1,3),(1,1),(0,1))
        self.tcn      = TemporalConv(out_ch, TCN_KERNEL, stride)
        self.drop_path = DropPath(drop_path_rate)
        self.relu     = nn.ReLU(inplace=True)

    def forward(self, x, A):
        x_gcn = self.gcn_bn(self.gcn(x, A))
        x_res = self.relu(self.drop_path(x_gcn) + self.res_conv(x))
        x_sp  = self.sp_pool(x_res)
        return self.relu(self.drop_path(self.tcn(x_sp)))


class StreamEncoder(nn.Module):
    def __init__(self, in_channels, adjacency, channels, strides, adaptive=True):
        super().__init__()
        K, V, _ = adjacency.shape
        self.register_buffer('A', torch.tensor(adjacency, dtype=torch.float32))
        self.input_bn   = nn.BatchNorm2d(in_channels)
        self.input_proj = nn.Sequential(
            nn.Conv2d(in_channels, channels[0], 1),
            nn.BatchNorm2d(channels[0]), nn.ReLU(inplace=True))
        n_blocks, blocks, prev = len(channels), [], channels[0]
        for i, (ch, st) in enumerate(zip(channels, strides)):
            dpr = DROP_PATH_RATE * i / max(1, n_blocks - 1)
            blocks.append(STGCNBlock(prev, ch, V, K, st, drop_path_rate=dpr, adaptive=adaptive))
            prev = ch
        self.blocks      = nn.ModuleList(blocks)
        self.edge_importance = nn.ParameterList([
            nn.Parameter(torch.ones(K, V, V)) for _ in self.blocks
        ])
        self.spatial_gap = nn.AdaptiveAvgPool2d((None, 1))

    def forward(self, x):
        x = self.input_proj(self.input_bn(x))
        early = None
        for i, blk in enumerate(self.blocks):
            x = blk(x, self.A * self.edge_importance[i])
            if i == SKIP_BLOCK_IDX:
                early = x
        return self.spatial_gap(x).squeeze(-1), early


class CrossStreamSkip(nn.Module):
    def __init__(self, early_ch, final_ch):
        super().__init__()
        self.proj = nn.Sequential(nn.Conv2d(early_ch, final_ch, 1),
                                  nn.BatchNorm2d(final_ch), nn.ReLU(inplace=True))
        self.spatial_gap = nn.AdaptiveAvgPool2d((None, 1))

    def forward(self, ej, em, ea):
        return self.spatial_gap(self.proj(ej + em + ea)).squeeze(-1)


class ConfidenceMasking(nn.Module):
    def __init__(self, conf_threshold=CONF_THRESH):
        super().__init__(); self.threshold = conf_threshold

    def forward(self, feats, x_joint):
        T_prime = feats[0].shape[-1]
        conf    = x_joint[:, 2, :, :].mean(dim=-1)
        conf_ds = F.adaptive_avg_pool1d(conf.unsqueeze(1).float(), T_prime).squeeze(1)
        mask    = (conf_ds > self.threshold).float().unsqueeze(1)
        return [f * mask for f in feats], mask


class ClassificationModule(nn.Module):
    def __init__(self, in_dim, hidden_dim=128, num_classes=2, dropout=0.5):
        super().__init__()
        self.fc1    = nn.Conv1d(in_dim, hidden_dim, 1)
        self.lrelu1 = nn.LeakyReLU(0.1, inplace=True)
        self.ln     = nn.LayerNorm(hidden_dim)
        self.lrelu2 = nn.LeakyReLU(0.1, inplace=True)
        self.drop   = nn.Dropout(p=dropout)
        self.fc_out = nn.Linear(hidden_dim, num_classes)

    def forward(self, x):
        x     = self.lrelu1(self.fc1(x))
        x     = self.lrelu2(self.ln(x.permute(0,2,1)).permute(0,2,1))
        x     = self.drop(x)
        x_avg = x.mean(dim=-1)
        return self.fc_out(x_avg)


class ThreeStreamSTGCN(nn.Module):
    def __init__(self, in_channels=IN_CHANNELS, num_classes=NUM_CLASSES,
                 skeleton_adj=None, channels=STREAM_CHANNELS,
                 strides=STREAM_STRIDES, dropout=DROPOUT):
        super().__init__()
        if skeleton_adj is None:
            skeleton_adj = SkeletonGraph().A
        self.stream_joint  = StreamEncoder(in_channels, skeleton_adj, channels, strides)
        self.stream_motion = StreamEncoder(in_channels, skeleton_adj, channels, strides)
        self.stream_angle  = StreamEncoder(in_channels, skeleton_adj, channels, strides)
        early_ch = channels[SKIP_BLOCK_IDX]; final_ch = channels[-1]
        self.cross_skip = CrossStreamSkip(early_ch, final_ch)
        self.classifier = ClassificationModule(4 * final_ch, 128, num_classes, dropout)

    def forward(self, x_joint, x_motion, x_angle):
        f1, ej = self.stream_joint(x_joint)
        f2, em = self.stream_motion(x_motion)
        f3, ea = self.stream_angle(x_angle)
        T_prime = f1.shape[-1]
        f_skip  = F.adaptive_avg_pool1d(self.cross_skip(ej, em, ea), T_prime)
        feat = torch.cat([f1, f2, f3, f_skip], dim=1)
        return self.classifier(feat)


class MultiStreamSTGCN(nn.Module):
    """Tổng quát hóa ThreeStreamSTGCN: chọn luồng ('J' joint, 'M' motion, 'T' luồng thứ ba = angle hoặc bone),
    bật/tắt adaptive graph và cross-stream skip. ('J','M','T'), True, True == ThreeStreamSTGCN gốc."""
    def __init__(self, streams=('J', 'M', 'T'), adaptive=True, use_skip=True,
                 in_channels=IN_CHANNELS, num_classes=NUM_CLASSES, channels=STREAM_CHANNELS,
                 strides=STREAM_STRIDES, dropout=DROPOUT):
        super().__init__()
        adj = SkeletonGraph().A
        self.streams = tuple(streams)
        self.encoders = nn.ModuleDict({s: StreamEncoder(in_channels, adj, channels, strides, adaptive)
                                       for s in self.streams})
        self.use_skip = use_skip
        early_ch, final_ch = channels[SKIP_BLOCK_IDX], channels[-1]
        if use_skip:
            self.skip_proj = nn.Sequential(nn.Conv2d(early_ch, final_ch, 1),
                                           nn.BatchNorm2d(final_ch), nn.ReLU(inplace=True))
            self.skip_gap = nn.AdaptiveAvgPool2d((None, 1))
        n_feat = len(self.streams) + int(use_skip)
        self.classifier = ClassificationModule(n_feat * final_ch, 128, num_classes, dropout)

    def forward(self, x_joint, x_motion, x_third):
        inputs = {'J': x_joint, 'M': x_motion, 'T': x_third}
        feats, earlies = [], []
        for s in self.streams:
            f, e = self.encoders[s](inputs[s]); feats.append(f); earlies.append(e)
        if self.use_skip:
            T_prime = feats[0].shape[-1]
            sk = self.skip_gap(self.skip_proj(torch.stack(earlies).sum(0))).squeeze(-1)
            feats.append(F.adaptive_avg_pool1d(sk, T_prime))
        return self.classifier(torch.cat(feats, dim=1))


# ── 2s-AGCN (Shi et al., CVPR 2019), cài đặt lại với cùng độ rộng kênh / số khối ────────────────
class AGCNGraphConv(nn.Module):
    # A_k (cố định) + B_k (học được, toàn cục) + C_k (attention phụ thuộc mẫu)
    def __init__(self, in_ch, out_ch, A, coff_embedding=4):
        super().__init__()
        K, V, _ = A.shape; inter = max(1, out_ch // coff_embedding)
        self.register_buffer('A', torch.tensor(A, dtype=torch.float32))
        self.PA = nn.Parameter(torch.full((K, V, V), 1e-6))
        self.conv_a = nn.ModuleList([nn.Conv2d(in_ch, inter, 1) for _ in range(K)])
        self.conv_b = nn.ModuleList([nn.Conv2d(in_ch, inter, 1) for _ in range(K)])
        self.conv_d = nn.ModuleList([nn.Conv2d(in_ch, out_ch, 1) for _ in range(K)])
        self.bn = nn.BatchNorm2d(out_ch)
        self.down = (nn.Sequential(nn.Conv2d(in_ch, out_ch, 1), nn.BatchNorm2d(out_ch))
                     if in_ch != out_ch else nn.Identity())
        self.K = K

    def forward(self, x):
        N, C_, T, V = x.shape
        y = 0
        for k in range(self.K):
            a1 = self.conv_a[k](x).permute(0, 3, 1, 2).reshape(N, V, -1)
            a2 = self.conv_b[k](x).reshape(N, -1, V)
            Ck = torch.softmax(torch.matmul(a1, a2) / a1.size(-1), dim=-2)
            Ak = self.A[k] + self.PA[k] + Ck
            y = y + self.conv_d[k](torch.matmul(x.reshape(N, C_ * T, V), Ak).reshape(N, C_, T, V))
        return F.relu(self.bn(y) + self.down(x))


class AGCNBlock(nn.Module):
    def __init__(self, in_ch, out_ch, A, stride=1):
        super().__init__()
        self.gcn = AGCNGraphConv(in_ch, out_ch, A)
        pad = (TCN_KERNEL - 1) // 2
        self.tcn = nn.Sequential(nn.Conv2d(out_ch, out_ch, (TCN_KERNEL, 1), (stride, 1), (pad, 0)),
                                 nn.BatchNorm2d(out_ch))
        self.res = (nn.Sequential(nn.Conv2d(in_ch, out_ch, 1, (stride, 1)), nn.BatchNorm2d(out_ch))
                    if (in_ch != out_ch or stride != 1) else nn.Identity())

    def forward(self, x):
        return F.relu(self.tcn(self.gcn(x)) + self.res(x))


class AGCNStream(nn.Module):
    def __init__(self, A, channels=STREAM_CHANNELS, strides=STREAM_STRIDES, num_classes=NUM_CLASSES):
        super().__init__()
        self.data_bn = nn.BatchNorm2d(IN_CHANNELS)
        layers, prev = [], IN_CHANNELS
        for ch, st in zip(channels, strides):
            layers.append(AGCNBlock(prev, ch, A, st)); prev = ch
        self.layers = nn.Sequential(*layers)
        self.drop = nn.Dropout(DROPOUT)
        self.fc = nn.Linear(prev, num_classes)

    def forward(self, x):
        x = self.layers(self.data_bn(x))
        return self.fc(self.drop(x.mean(dim=(2, 3))))


class TwoStreamAGCN(nn.Module):
    # Bản gốc huấn luyện 2 luồng riêng rồi cộng điểm; ở đây huấn luyện đồng thời và cộng logit
    def __init__(self):
        super().__init__()
        A = SkeletonGraph().A
        self.joint, self.bone = AGCNStream(A), AGCNStream(A)

    def forward(self, x_joint, x_motion, x_third):     # x_third = vector xương (THIRD_STREAM='bone')
        return self.joint(x_joint) + self.bone(x_third)


class RNNBaseline(nn.Module):
    def __init__(self, cell='lstm', hidden=128, layers=2):
        super().__init__()
        D = IN_CHANNELS * NUM_KEYPOINTS
        self.in_bn = nn.BatchNorm1d(D)
        rnn = nn.LSTM if cell == 'lstm' else nn.GRU
        self.rnn = rnn(D, hidden, layers, batch_first=True, dropout=0.3)
        self.drop = nn.Dropout(DROPOUT)
        self.fc = nn.Linear(hidden, NUM_CLASSES)

    def forward(self, x_joint, x_motion, x_third):
        N, C_, T, V = x_joint.shape
        x = self.in_bn(x_joint.permute(0, 1, 3, 2).reshape(N, C_ * V, T)).permute(0, 2, 1)
        out, _ = self.rnn(x)
        return self.fc(self.drop(out.mean(dim=1)))


VARIANTS = {
    'full':           dict(kind='stgcn', streams='JMT', third='angle', adaptive=True,  skip=True),
    'no_skip':        dict(kind='stgcn', streams='JMT', third='angle', adaptive=True,  skip=False),
    'no_adaptive':    dict(kind='stgcn', streams='JMT', third='angle', adaptive=False, skip=True),
    'JMB':            dict(kind='stgcn', streams='JMT', third='bone',  adaptive=True,  skip=True),
    'JM':             dict(kind='stgcn', streams='JM',  third='angle', adaptive=True,  skip=True),
    'JA':             dict(kind='stgcn', streams='JT',  third='angle', adaptive=True,  skip=True),
    'MA':             dict(kind='stgcn', streams='MT',  third='angle', adaptive=True,  skip=True),
    'J':              dict(kind='stgcn', streams='J',   third='angle', adaptive=True,  skip=True),
    'M':              dict(kind='stgcn', streams='M',   third='angle', adaptive=True,  skip=True),
    'A':              dict(kind='stgcn', streams='T',   third='angle', adaptive=True,  skip=True),
    'JM_no_adaptive': dict(kind='stgcn', streams='JM',  third='angle', adaptive=False, skip=True),
    'stgcn':          dict(kind='stgcn', streams='J',   third='angle', adaptive=False, skip=False),
    '2s_agcn':        dict(kind='agcn',  third='bone'),
    'lstm':           dict(kind='rnn',   cell='lstm',   third='angle'),
    'gru':            dict(kind='rnn',   cell='gru',    third='angle'),
}


def build_model(variant):
    v = VARIANTS[variant]
    if v['kind'] == 'stgcn':
        return MultiStreamSTGCN(tuple(v['streams']), v['adaptive'], v['skip'])
    if v['kind'] == 'agcn':
        return TwoStreamAGCN()
    return RNNBaseline(v['cell'])


def n_params(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


# Kiểm tra: biến thể 'full' có đúng số tham số như ThreeStreamSTGCN gốc
assert n_params(build_model('full')) == n_params(ThreeStreamSTGCN()), 'full khác mô hình gốc!'
for k in VARIANTS:
    print(f'{k:<15} {n_params(build_model(k))/1e6:.3f}M tham số')

## 6. Augmentation và Dataset (thêm lựa chọn luồng thứ ba: angle hoặc bone thô)

In [ ]:
THIRD_STREAM = 'angle'   # được run_one đặt theo biến thể


def compute_bone_np(j):
    # Vector xương thô (dx, dy) từ khớp cha tới khớp con, chưa chuẩn hóa độ dài; kênh 3 = min confidence
    F_, V, _ = j.shape; b = np.zeros((F_, V, 3), dtype=np.float32)
    for v in range(V):
        p = PARENT_MAP_AUG[v]
        if p is None:
            b[:, v, 2] = j[:, v, 2]
        else:
            b[:, v, :2] = j[:, v, :2] - j[:, p, :2]
            b[:, v, 2] = np.minimum(j[:, v, 2], j[:, p, 2])
    return b


PARENT_MAP_AUG = {
    13:None, 7:13, 8:13, 1:7, 2:8, 0:1,
    3:1, 4:2, 5:3, 6:4, 9:7, 10:8, 11:9, 12:10,
}

def compute_motion_np(j):
    m = np.zeros_like(j); m[1:] = j[1:]-j[:-1]; m[:,:,2]=j[:,:,2]; return m

def compute_angle_np(j):
    F,V,_ = j.shape; angle = np.zeros((F,V,3),dtype=np.float32)
    for v in range(V):
        p = PARENT_MAP_AUG[v]
        if p is None:
            angle[:,v,0]=1.0; angle[:,v,2]=j[:,v,2]
        else:
            dy=j[:,v,1]-j[:,p,1]; dx=j[:,v,0]-j[:,p,0]
            th=np.arctan2(dy,dx)
            angle[:,v,0]=np.cos(th); angle[:,v,1]=np.sin(th)
            angle[:,v,2]=np.minimum(j[:,v,2],j[:,p,2])
    return angle

def aug_rotate(j, deg):
    theta=deg*np.pi/180; c,s=np.cos(theta),np.sin(theta)
    R=np.array([[c,-s],[s,c]],dtype=np.float32)
    j=j.copy(); pivot=j[:,13:14,:2]
    j[:,:,:2]=(j[:,:,:2]-pivot)@R.T+pivot; return j

def aug_zoom(j, factor):
    j=j.copy(); pivot=j[:,13:14,:2]
    j[:,:,:2]=(j[:,:,:2]-pivot)*factor+pivot; return j

def aug_mask_joints(j, mask_prob=0.15):
    j=j.copy(); mask=np.random.rand(j.shape[1])<mask_prob
    j[:,mask,2]=0.0; return j

def aug_hflip(j, img_w):
    j=j.copy(); j[:,:,0]=img_w-j[:,:,0]
    for l,r in FLIP_PAIRS: j[:,[l,r],:]=j[:,[r,l],:]; return j

def aug_noise(j, img_w, ratio=0.005):
    j=j.copy()
    j[:,:,:2]+=np.random.normal(0,ratio*img_w,j[:,:,:2].shape).astype(np.float32); return j

def aug_speed(j, factor):
    F=len(j); new_F=max(8,int(F/factor))
    ot=np.linspace(0,1,F); nt=np.linspace(0,1,new_F)
    out=np.zeros((new_F,j.shape[1],3),dtype=np.float32)
    for v in range(j.shape[1]):
        for c in range(3): out[:,v,c]=np.interp(nt,ot,j[:,v,c])
    if new_F>=SEQ_LEN: return out[:SEQ_LEN]
    pad=np.zeros((SEQ_LEN-new_F,j.shape[1],3),dtype=np.float32)
    return np.concatenate([out,pad])

def aug_crop(j, img_w, img_h, ratio=0.85):
    j=j.copy(); wc=int(img_w*ratio); hc=int(img_h*ratio)
    x0=np.random.randint(0,max(1,img_w-wc)); y0=np.random.randint(0,max(1,img_h-hc))
    in_box=((j[:,:,0]>=x0)&(j[:,:,0]<x0+wc)&(j[:,:,1]>=y0)&(j[:,:,1]<y0+hc))
    j[:,:,2]=np.where(in_box,j[:,:,2],0.0)
    j[:,:,0]=np.clip(j[:,:,0]-x0,0,wc-1); j[:,:,1]=np.clip(j[:,:,1]-y0,0,hc-1); return j

def augment_online(j, img_w, img_h, label):
    is_fall = (label == 1)
    if np.random.rand() < (0.55 if is_fall else 0.50):
        j = aug_mask_joints(j, np.random.uniform(0.05, 0.22 if is_fall else 0.18))
    if np.random.rand() < (0.75 if is_fall else 0.65):
        j = aug_rotate(j, np.random.uniform(-25 if is_fall else -20, 25 if is_fall else 20))
    if np.random.rand() < (0.70 if is_fall else 0.60):
        lo,hi = (0.78,1.22) if is_fall else (0.82,1.18)
        j = aug_zoom(j, np.random.uniform(lo, hi))
    if np.random.rand() < 0.5:
        j = aug_hflip(j, img_w)
    if np.random.rand() < (0.45 if is_fall else 0.40):
        j = aug_noise(j, img_w, np.random.uniform(0.003, 0.014 if is_fall else 0.010))
    if np.random.rand() < (0.45 if is_fall else 0.25):
        j = aug_speed(j, np.random.uniform(0.70,1.35) if is_fall else np.random.uniform(0.85,1.15))
    if np.random.rand() < (0.35 if is_fall else 0.20):
        j = aug_crop(j, img_w, img_h, np.random.uniform(0.80, 0.95))
    return j


class CustomFallDataset(Dataset):
    def __init__(self, X_joint, y, img_w, img_h, category, indices, augment=False):
        self.X_joint  = X_joint
        self.y        = y
        self.img_w    = img_w
        self.img_h    = img_h
        self.category = category
        self.indices  = indices
        self.augment  = augment

    def get_labels(self):
        return self.y[self.indices]

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx):
        gi  = self.indices[idx]
        j   = self.X_joint[gi].copy()
        y   = int(self.y[gi])
        iw  = int(self.img_w[gi])
        ih  = int(self.img_h[gi])
        cat = str(self.category[gi])

        if self.augment:
            j = augment_online(j, iw, ih, y)

        if COORD_MODE == 'unit':          # ablation R2.8: chuẩn hóa theo kích thước ảnh
            j = j.copy(); j[:, :, 0] /= iw; j[:, :, 1] /= ih

        m = compute_motion_np(j)
        a = compute_angle_np(j) if THIRD_STREAM == 'angle' else compute_bone_np(j)

        j = torch.FloatTensor(j.transpose(2, 0, 1))
        m = torch.FloatTensor(m.transpose(2, 0, 1))
        a = torch.FloatTensor(a.transpose(2, 0, 1))

        cat_label = CATEGORY_LABEL.get(cat, 0)
        return j, m, a, torch.tensor(y, dtype=torch.long), cat, cat_label


def build_weighted_sampler(dataset):
    labels      = dataset.get_labels().astype(np.int64)
    categories  = dataset.category[dataset.indices]
    cat_indices = np.array([CATEGORY_LABEL.get(c, 0) for c in categories])
    n_cats      = len(CATEGORY_LABEL)
    cat_counts  = np.bincount(cat_indices, minlength=n_cats)

    # [FIX] đồng nhất sqrt inverse với FocalLoss alpha
    cat_counts_safe = np.where(cat_counts == 0, 1, cat_counts)
    weight_per_cat  = 1.0 / np.sqrt(cat_counts_safe.astype(np.float64))
    sample_weights  = torch.DoubleTensor([weight_per_cat[ci] for ci in cat_indices])

    total      = len(labels)
    fall_total = int(labels.sum())
    log(f"  Train category dist: " +
        " | ".join(f"{k}={cat_counts[v]}" for k, v in CATEGORY_LABEL.items()))
    log(f"  Train binary: fall={fall_total} ({100*fall_total/total:.1f}%)  "
        f"non_fall={total-fall_total} ({100*(total-fall_total)/total:.1f}%)")
    log(f"  Sampler weights (sqrt-inv): " +
        " | ".join(f"{k}={weight_per_cat[v]:.4f}" for k, v in CATEGORY_LABEL.items()))

    return WeightedRandomSampler(sample_weights, len(labels), replacement=True)


# ── [FIX-A] Same-label Mixup ──────────────────────────────────────────────────
def mixup_data(j, m, a, y, alpha=MIXUP_ALPHA):
    """
    Ưu tiên mix cùng class (same-label mixup).
    Khi dữ liệu mất cân bằng, cross-class mixup tạo ra vùng ambiguous
    gần decision boundary, làm mô hình khó học fall minority class.
    Same-label mixup tăng diversity mà không làm mờ boundary.

    Fallback: nếu class chỉ có 1 sample trong batch → dùng random như cũ.
    """
    if alpha <= 0 or np.random.rand() > 0.5:
        return j, m, a, y, None

    lam = float(np.random.beta(alpha, alpha))
    B   = j.size(0)

    # Xây dựng chỉ số mix ưu tiên cùng class
    same_class_idx = torch.zeros(B, dtype=torch.long, device=j.device)
    for i in range(B):
        yi         = y[i].item()
        candidates = (y == yi).nonzero(as_tuple=True)[0]
        # Loại bỏ chính nó nếu có đủ candidate
        candidates = candidates[candidates != i]
        if len(candidates) > 0:
            pick = candidates[torch.randint(len(candidates), (1,)).item()]
        else:
            # Fallback: random bất kỳ (batch quá nhỏ hoặc class duy nhất)
            pool = torch.arange(B, device=j.device)
            pool = pool[pool != i]
            pick = pool[torch.randint(len(pool), (1,)).item()] if len(pool) > 0 else i
        same_class_idx[i] = pick

    idx = same_class_idx
    j_mix = lam * j + (1 - lam) * j[idx]
    m_mix = lam * m + (1 - lam) * m[idx]
    a_mix = lam * a + (1 - lam) * a[idx]

    return j_mix, m_mix, a_mix, y, (y[idx], lam)
# ─────────────────────────────────────────────────────────────────────────────

## 7. Huấn luyện và đánh giá (giữ nguyên)

In [ ]:
def mixup_criterion(criterion, logits, y, mixed_info):
    if mixed_info is None:
        return criterion(logits, y)
    y_b, lam = mixed_info
    return lam*criterion(logits, y) + (1-lam)*criterion(logits, y_b)


def train_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total_loss = 0.0; preds_all = []; labels_all = []
    for j, m, a, y, _cat, _cl in loader:
        j, m, a, y = j.to(device), m.to(device), a.to(device), y.to(device)
        j, m, a, y, mi = mixup_data(j, m, a, y)
        optimizer.zero_grad()
        logits = model(j, m, a)
        loss   = mixup_criterion(criterion, logits, y, mi)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
        optimizer.step()
        total_loss += loss.item()
        preds_all.extend(logits.argmax(1).cpu().numpy())
        labels_all.extend(y.cpu().numpy())
    acc = accuracy_score(labels_all, preds_all)
    pr_t, rc_t, _, _ = precision_recall_fscore_support(
        labels_all, preds_all, average='binary', zero_division=0)
    denom_f2 = 4 * pr_t + rc_t
    f2 = (5 * pr_t * rc_t / denom_f2) if denom_f2 > 0 else 0.0
    return total_loss / len(loader), acc, f2


@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss = 0.0; preds_all = []; labels_all = []; probs_all = []
    cats_all = []; cat_labels_all = []
    for j, m, a, y, cat, cat_label in loader:
        j, m, a, y = j.to(device), m.to(device), a.to(device), y.to(device)
        logits = model(j, m, a)
        total_loss += criterion(logits, y).item()
        preds_all.extend(logits.argmax(1).cpu().numpy())
        labels_all.extend(y.cpu().numpy())
        probs_all.extend(torch.softmax(logits, 1)[:, 1].cpu().numpy())
        cats_all.extend(list(cat))
        cat_labels_all.extend(cat_label.numpy().tolist())
    acc = accuracy_score(labels_all, preds_all)
    prec, rec, f1, _ = precision_recall_fscore_support(
        labels_all, preds_all, average='binary', zero_division=0)
    cm = confusion_matrix(labels_all, preds_all)
    return (total_loss / len(loader), acc, prec, rec, f1,
            cm, preds_all, labels_all, probs_all, cats_all, cat_labels_all)


def eval_at_threshold(labels, probs, thr):
    preds  = (np.array(probs)  >= thr).astype(int)
    labels = np.array(labels)
    if len(np.unique(labels)) < 2:
        tp = int(((preds==1)&(labels==1)).sum()); fp = int(((preds==1)&(labels==0)).sum())
        fn = int(((preds==0)&(labels==1)).sum()); tn = int(((preds==0)&(labels==0)).sum())
    else:
        tn, fp, fn, tp = (int(v) for v in confusion_matrix(labels, preds).ravel())
    rec  = tp/(tp+fn) if (tp+fn)>0 else 0.0
    spec = tn/(tn+fp) if (tn+fp)>0 else 0.0
    prec = tp/(tp+fp) if (tp+fp)>0 else 0.0
    f1   = 2*prec*rec/(prec+rec) if (prec+rec)>0 else 0.0
    return dict(acc=(tp+tn)/(tp+tn+fp+fn), prec=prec, rec=rec,
                spec=spec, f1=f1, mr=1-rec,
                fpr=fp/(fp+tn) if (fp+tn)>0 else 0.0,
                tp=tp, fp=fp, fn=fn, tn=tn)


def eval_per_category(labels_all, probs_all, cats_all, thr=0.5):
    """
    One-vs-rest confusion cho 4 category:
      non_fall, Lying_Falls, Sitting_Falls, Standing_Falls

    Lưu ý:
    - Model là binary (fall / non_fall), nên với 3 fall-category:
      pred_positive được hiểu là (pred == fall).
    """
    probs_arr = np.array(probs_all)
    cats_arr  = np.array(cats_all)
    preds_arr = (probs_arr >= thr).astype(int)  # 1=fall, 0=non_fall

    results = {}
    for cat in ALL_CATEGORIES:
        true_pos = (cats_arr == cat)

        if cat == "non_fall":
            pred_pos = (preds_arr == 0)  # dự đoán non_fall
        else:
            pred_pos = (preds_arr == 1)  # dự đoán fall (không phân subtype)

        tp = int(( pred_pos &  true_pos).sum())
        fp = int(( pred_pos & ~true_pos).sum())
        fn = int((~pred_pos &  true_pos).sum())
        tn = int((~pred_pos & ~true_pos).sum())

        prec = tp / (tp + fp) if (tp + fp) > 0 else 0.0
        rec  = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        spec = tn / (tn + fp) if (tn + fp) > 0 else 0.0
        f1   = 2 * prec * rec / (prec + rec) if (prec + rec) > 0 else 0.0

        results[cat] = dict(
            precision=prec,
            rec=rec,
            spec=spec,
            f1=f1,
            mr=1 - rec,
            tp=tp, fp=fp, fn=fn, tn=tn,
            n=int(true_pos.sum()),
            support=int(true_pos.sum()),
        )

    return results

## 8. Một lượt chạy (như notebook 02, thêm tham số biến thể)

In [ ]:
def youden_threshold(labels, probs):
    if len(set(labels)) < 2:
        return INFERENCE_THR
    fpr, tpr, thr = roc_curve(labels, probs)
    return float(thr[int(np.argmax(tpr - fpr))])


def metric_block(labels, probs, thr):
    labels = np.asarray(labels); probs = np.asarray(probs)
    r = eval_at_threshold(labels, probs, thr)
    r['bal_acc'] = 0.5 * (r['rec'] + r['spec'])
    two = len(set(labels.tolist())) > 1
    r['roc_auc'] = float(roc_auc_score(labels, probs)) if two else float('nan')
    if two:
        pc, rc, _ = precision_recall_curve(labels, probs); r['pr_auc'] = float(auc(rc, pc))
    else:
        r['pr_auc'] = float('nan')
    r['thr'] = float(thr); r['n'] = int(len(labels))
    return {k: (float(v) if isinstance(v, (float, np.floating)) else v) for k, v in r.items()}


def to_video_level(df):
    agg = df.groupby('video_id').agg(prob=('prob', VIDEO_AGG), label=('label', 'first'),
                                     subject=('subject', 'first'), view=('view', 'first'),
                                     category=('category', 'first'), nonfall_type=('nonfall_type', 'first'),
                                     orientation=('orientation', 'first'), n_windows=('prob', 'size'))
    return agg.reset_index()


def run_one(protocol, fold_name, seed, d, tr, va, te, variant='full', save_root=None):
    save_dir = os.path.join(save_root or RESULTS_DIR, protocol, fold_name, f'seed_{seed}')
    mpath = os.path.join(save_dir, 'metrics.json')
    if os.path.exists(mpath):
        log(f'  bỏ qua (đã có): {protocol}/{fold_name}/seed_{seed}')
        return json.load(open(mpath))
    os.makedirs(save_dir, exist_ok=True)
    set_global_seed(seed)
    global THIRD_STREAM
    THIRD_STREAM = VARIANTS[variant].get('third', 'angle')

    X_joint, y, img_w, img_h, category = d['X_joint'], d['y'], d['img_w'], d['img_h'], d['category']
    train_ds = CustomFallDataset(X_joint, y, img_w, img_h, category, tr, augment=True)
    val_ds   = CustomFallDataset(X_joint, y, img_w, img_h, category, va, augment=False)
    test_ds  = CustomFallDataset(X_joint, y, img_w, img_h, category, te, augment=False)

    cat_counts_4 = np.bincount([CATEGORY_LABEL[c] for c in category[tr]], minlength=4).astype(np.float32)
    w4 = 1.0 / np.sqrt(cat_counts_4.clip(min=1))
    fall_w = float(np.average(w4[1:], weights=cat_counts_4[1:].clip(min=1))) * FALL_ALPHA_BOOST
    alpha_w = torch.FloatTensor([float(w4[0]), fall_w]); alpha_w = (alpha_w / alpha_w.sum()).to(DEVICE)

    sampler      = build_weighted_sampler(train_ds)
    train_loader = DataLoader(train_ds, BATCH_SIZE, sampler=sampler, num_workers=2, pin_memory=True)
    val_loader   = DataLoader(val_ds,   BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
    test_loader  = DataLoader(test_ds,  BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

    model     = build_model(variant).to(DEVICE)
    criterion = FocalLoss(FOCAL_GAMMA, alpha_w, label_smoothing=LABEL_SMOOTHING)
    optimizer = optim.AdamW(model.parameters(), LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    scheduler = build_scheduler(optimizer, WARMUP_EPOCHS, EPOCHS)

    best_loss = float('inf'); best_va_f2 = -1.0; plateau_cnt = 0; best_ep = 1
    model_path = os.path.join(save_dir, 'best_model.pth')
    history = dict(epoch=[], train_loss=[], val_loss=[], train_f2=[], val_f2=[])
    for epoch in range(EPOCHS):
        tr_loss, _, tr_f2 = train_epoch(model, train_loader, criterion, optimizer, DEVICE)
        scheduler.step()
        va_loss, _, va_p, va_r, _, *_ = evaluate(model, val_loader, criterion, DEVICE)
        va_f2 = (5 * va_p * va_r / (4 * va_p + va_r)) if (4 * va_p + va_r) > 0 else 0.0
        for k, v in zip(history, [epoch + 1, tr_loss, va_loss, tr_f2, va_f2]):
            history[k].append(float(v))
        if (va_loss < best_loss - PLATEAU_DELTA) or (va_f2 > best_va_f2 + PLATEAU_DELTA):
            best_loss, best_va_f2, plateau_cnt, best_ep = va_loss, va_f2, 0, epoch + 1
            torch.save(model.state_dict(), model_path)
        else:
            plateau_cnt += 1
        if (epoch + 1) % 25 == 0:
            print(f'    ep{epoch+1:>3}  tr_f2={tr_f2:.4f}  va_f2={va_f2:.4f}')
        if epoch + 1 >= MIN_EPOCHS and plateau_cnt >= PLATEAU_PATIENCE:
            log(f'    Early stop @ ep{epoch+1}  (best={best_ep})'); break
    model.load_state_dict(torch.load(model_path, weights_only=True))

    def frame_of(idx, probs):
        return pd.DataFrame(dict(idx=idx, video_id=d['video_id'][idx], window_idx=d['window_idx'][idx],
                                 subject=d['subject'][idx], view=d['view'][idx], category=d['category'][idx],
                                 nonfall_type=d['nonfall_type'][idx], orientation=d['orientation'][idx],
                                 is_flip=d['is_flip'][idx], label=y[idx], prob=np.asarray(probs, float)))

    *_, va_labels, va_probs, _, _ = evaluate(model, val_loader, criterion, DEVICE)
    *_, te_labels, te_probs, _, _ = evaluate(model, test_loader, criterion, DEVICE)
    va_df, te_df = frame_of(va, va_probs), frame_of(te, te_probs)
    assert (te_df.label.values == np.asarray(te_labels)).all()

    thr_w = youden_threshold(va_df.label.values, va_df.prob.values)
    va_v, te_v = to_video_level(va_df), to_video_level(te_df)
    thr_v = youden_threshold(va_v.label.values, va_v.prob.values)
    te_df['thr'] = thr_w; te_df['pred'] = (te_df.prob >= thr_w).astype(int)
    te_v['thr'] = thr_v;  te_v['pred'] = (te_v.prob >= thr_v).astype(int)
    te_df.to_csv(os.path.join(save_dir, 'predictions_window.csv'), index=False)
    te_v.to_csv(os.path.join(save_dir, 'predictions_video.csv'), index=False)

    result = dict(protocol=protocol, variant=variant, n_params=n_params(model), fold=fold_name, seed=seed, best_ep=best_ep, coord_mode=COORD_MODE,
                  n_train=int(len(tr)), n_val=int(len(va)), n_test=int(len(te)),
                  window=metric_block(te_df.label, te_df.prob, thr_w),
                  video=metric_block(te_v.label, te_v.prob, thr_v),
                  history=history)
    json.dump(result, open(mpath, 'w'), indent=2)
    w, v = result['window'], result['video']
    log(f"  {protocol}/{fold_name}/seed_{seed}: WIN acc={w['acc']:.4f} rec={w['rec']:.4f} spec={w['spec']:.4f} "
        f"| VID acc={v['acc']:.4f} rec={v['rec']:.4f} spec={v['spec']:.4f}", 'GOOD')
    return result

## 9. Chạy các biến thể

`full` không train lại: notebook đọc kết quả E1 của notebook 02 trong `results_v2/<COORD_MODE>/E1/`.
Nếu thư mục đó chưa có, `full` sẽ được train ở đây như các biến thể khác.

In [ ]:
DRY_RUN = False

def variant_root(variant):
    if variant == 'full' and os.path.isdir(os.path.join(RESULTS_DIR, 'E1')):
        return RESULTS_DIR                       # dùng lại kết quả E1 của notebook 02
    return os.path.join(ABL_DIR, variant)

folds = make_folds('E1', data)
for tr_, va_, te_ in [(f[1], f[2], f[3]) for f in folds]:
    assert_no_leak('E1', data, tr_, va_, te_)

for variant in VARIANTS_TO_RUN:
    seeds = SEEDS_TIER_B if variant in TIER_B else SEEDS
    sep('═'); log(f'VARIANT {variant}  {VARIANTS[variant]}  seeds={seeds}'); sep('═')
    if DRY_RUN:
        continue
    for fold_name, tr, va, te in folds:
        for seed in seeds:
            run_one('E1', fold_name, seed, data, tr, va, te, variant=variant, save_root=variant_root(variant))
            torch.cuda.empty_cache()

## 10. Tổng hợp, kiểm định thống kê, số tham số và thời gian suy luận

* Chỉ số gộp 3 fold (mức video), trung bình ± std trên seed — cùng cách tính với notebook 02.
* **McNemar** (chính xác, nhị thức) so với `full` trên 655 video, ghép cặp theo cùng seed; báo cáo b (full đúng,
  biến thể sai), c (full sai, biến thể đúng) cộng dồn 5 seed và p trung vị qua các seed.
* **Wilcoxon** ghép cặp trên accuracy của 15 lượt (fold × seed).
* Thời gian suy luận: batch = 1, một cửa sổ 60 frame, trên CPU của Colab (chỉ để so tương đối giữa mô hình).

In [ ]:
from scipy.stats import binomtest, wilcoxon
import time as _time

SUM_DIR = os.path.join(ABL_DIR, 'summary'); os.makedirs(SUM_DIR, exist_ok=True)

def load_variant(variant):
    root = os.path.join(variant_root(variant), 'E1')
    runs = []
    if not os.path.isdir(root):
        return runs
    for fold in sorted(os.listdir(root)):
        for sd in sorted(os.listdir(os.path.join(root, fold))):
            p = os.path.join(root, fold, sd)
            if os.path.exists(os.path.join(p, 'metrics.json')):
                runs.append(dict(fold=fold, seed=sd, metrics=json.load(open(os.path.join(p, 'metrics.json'))),
                                 preds=pd.read_csv(os.path.join(p, 'predictions_video.csv'))))
    return runs

def pooled(df):
    lab, pred = df.label.values, df.pred.values
    tp = ((pred == 1) & (lab == 1)).sum(); fp = ((pred == 1) & (lab == 0)).sum()
    fn = ((pred == 0) & (lab == 1)).sum(); tn = ((pred == 0) & (lab == 0)).sum()
    rec, spec = tp / (tp + fn), tn / (tn + fp); prec = tp / max(1, tp + fp)
    return dict(acc=(tp + tn) / len(lab), rec=rec, spec=spec, f1=2 * prec * rec / max(1e-9, prec + rec),
                roc_auc=roc_auc_score(lab, df.prob.values), tp=tp, fp=fp, fn=fn, tn=tn)

def fmt(x):
    x = np.asarray(x, float); return f'{100*x.mean():.2f} ± {100*x.std():.2f}'

order = ['full'] + [v for v in VARIANTS if v != 'full']
loaded = {v: load_variant(v) for v in order}
loaded = {v: r for v, r in loaded.items() if r}
assert 'full' in loaded, 'Chưa có kết quả full (E1 của notebook 02)'

full_runs = {(r['fold'], r['seed']): r for r in loaded['full']}
rows = []
for v, runs in loaded.items():
    allp = pd.concat([r['preds'].assign(fold=r['fold'], seed=r['seed']) for r in runs])
    per_seed = [pooled(g) for _, g in allp.groupby('seed')]
    row = dict(variant=v, n_runs=len(runs), n_params=n_params(build_model(v)))
    for k in ['acc', 'rec', 'spec', 'f1', 'roc_auc']:
        row[k] = fmt([p[k] for p in per_seed])
    row['TP/FP/FN/TN (TB/seed)'] = '/'.join(f"{np.mean([p[k] for p in per_seed]):.1f}" for k in ['tp', 'fp', 'fn', 'tn'])
    if v != 'full':
        b = c = 0; ps = []
        for sd, g in allp.groupby('seed'):
            f = pd.concat([full_runs[(fo, sd)]['preds'] for fo in g.fold.unique() if (fo, sd) in full_runs])
            mrg = g.merge(f[['video_id', 'pred', 'label']], on='video_id', suffixes=('', '_full'))
            if len(mrg) == 0:
                continue
            ok_v, ok_f = mrg.pred == mrg.label, mrg.pred_full == mrg.label
            bs, cs = int((ok_f & ~ok_v).sum()), int((~ok_f & ok_v).sum())
            b += bs; c += cs
            ps.append(binomtest(min(bs, cs), bs + cs, 0.5).pvalue if bs + cs > 0 else 1.0)
        pairs = [(full_runs[(r['fold'], r['seed'])]['metrics']['video']['acc'], r['metrics']['video']['acc'])
                 for r in runs if (r['fold'], r['seed']) in full_runs]
        try:
            pw = wilcoxon([a for a, _ in pairs], [x for _, x in pairs]).pvalue if len(pairs) >= 6 else float('nan')
        except ValueError:
            pw = 1.0
        row.update({'McNemar b/c (cộng 5 seed)': f'{b}/{c}', 'McNemar p (trung vị)': np.median(ps) if ps else np.nan,
                    'Wilcoxon p': pw})
    rows.append(row)

table = pd.DataFrame(rows).set_index('variant')
display(table)
table.to_csv(os.path.join(SUM_DIR, 'ablation_E1_video.csv'))
print('->', os.path.join(SUM_DIR, 'ablation_E1_video.csv'))

In [ ]:
# Thời gian suy luận CPU, batch = 1 (R2.9; số tuyệt đối phụ thuộc máy, dùng để so tương đối)
lat = []
x = torch.randn(1, IN_CHANNELS, SEQ_LEN, NUM_KEYPOINTS)
for v in loaded:
    mdl = build_model(v).eval()
    with torch.no_grad():
        for _ in range(5): mdl(x, x, x)
        t0 = _time.perf_counter()
        for _ in range(50): mdl(x, x, x)
    lat.append(dict(variant=v, params_M=n_params(mdl) / 1e6, cpu_ms_per_window=(_time.perf_counter() - t0) / 50 * 1000))
lat = pd.DataFrame(lat).set_index('variant').round(3)
display(lat); lat.to_csv(os.path.join(SUM_DIR, 'params_latency_cpu.csv'))